In [7]:
import sys
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nCurrent working directory:")
print(Path.cwd())

import numpy as np
import pandas as pd
import torch
import cv2

print("\nNumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())

Python executable:
d:\multimodal_emotion_project\.venv\Scripts\python.exe

Python version:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]

Current working directory:
d:\multimodal_emotion_project\notebooks

NumPy: 2.4.6
Pandas: 3.1.0
PyTorch: 2.14.1+cpu
OpenCV: 4.10.0
CUDA available: False


In [8]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"D:\multimodal_emotion_project")
DATA_DIR = PROJECT_DIR / "data" / "processed"
MANIFEST_PATH = DATA_DIR / "manifest.csv"

print("Project exists:", PROJECT_DIR.exists())
print("Processed data exists:", DATA_DIR.exists())
print("Manifest exists:", MANIFEST_PATH.exists())

if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)

    print("\nDataset shape:", manifest.shape)
    print("\nColumn names:")
    print(manifest.columns.tolist())

    print("\nFirst five rows:")
    display(manifest.head())

    print("\nNumber of clips per emotion:")
    display(manifest["emotion"].value_counts().sort_index())

    print("\nPreprocessing status:")
    display(manifest["preprocessing_status"].value_counts())
else:
    print("\nManifest not found. Check your project folder path.")

Project exists: True
Processed data exists: True
Manifest exists: True

Dataset shape: (360, 13)

Column names:
['emotion_code', 'emotion', 'actor_id', 'gender', 'statement', 'intensity', 'repetition', 'clip_id', 'video_path', 'audio_feature', 'video_feature', 'preprocessing_status', 'error']

First five rows:


,emotion_code,emotion,actor_id,gender,statement,intensity,repetition,clip_id,video_path,audio_feature,video_feature,preprocessing_status,error
0,1,neutral,1,male,1,1,1,01-01-01-01-01-01-01,data\raw\RAVDESS\Video_Speech_Actor_01\Actor_0...,data\processed\audio\01-01-01-01-01-01-01.npy,data\processed\video\01-01-01-01-01-01-01.npy,ok,NaN
1,1,neutral,1,male,1,1,2,01-01-01-01-01-02-01,data\raw\RAVDESS\Video_Speech_Actor_01\Actor_0...,data\processed\audio\01-01-01-01-01-02-01.npy,data\processed\video\01-01-01-01-01-02-01.npy,ok,NaN
2,1,neutral,1,male,2,1,1,01-01-01-01-02-01-01,data\raw\RAVDESS\Video_Speech_Actor_01\Actor_0...,data\processed\audio\01-01-01-01-02-01-01.npy,data\processed\video\01-01-01-01-02-01-01.npy,ok,NaN
3,1,neutral,1,male,2,1,2,01-01-01-01-02-02-01,data\raw\RAVDESS\Video_Speech_Actor_01\Actor_0...,data\processed\audio\01-01-01-01-02-02-01.npy,data\processed\video\01-01-01-01-02-02-01.npy,ok,NaN
4,2,calm,1,male,1,1,1,01-01-02-01-01-01-01,data\raw\RAVDESS\Video_Speech_Actor_01\Actor_0...,data\processed\audio\01-01-02-01-01-01-01.npy,data\processed\video\01-01-02-01-01-01-01.npy,ok,NaN



Number of clips per emotion:


emotion
angry        48
calm         48
disgust      48
fearful      48
happy        48
neutral      24
sad          48
surprised    48
Name: count, dtype: int64


Preprocessing status:


preprocessing_status
ok    360
Name: count, dtype: int64

In [9]:
from pathlib import Path

if MANIFEST_PATH.exists():
    missing_audio = []
    missing_video = []

    for _, row in manifest.iterrows():
        for column, missing_list in [
            ("audio_feature", missing_audio),
            ("video_feature", missing_video),
        ]:
            if column not in manifest.columns:
                continue

            feature_path = Path(str(row[column]))

            if not feature_path.is_absolute():
                feature_path = PROJECT_DIR / feature_path

            if not feature_path.exists():
                missing_list.append(str(feature_path))

    print("Total clips:", len(manifest))
    print("Missing audio features:", len(missing_audio))
    print("Missing video features:", len(missing_video))

    if missing_audio:
        print("\nExample missing audio path:", missing_audio[0])

    if missing_video:
        print("\nExample missing video path:", missing_video[0])

Total clips: 360
Missing audio features: 0
Missing video features: 0


In [10]:
from pathlib import Path

SRC_DIR = PROJECT_DIR / "src"

print("Source folder exists:", SRC_DIR.exists())

if SRC_DIR.exists():
    print("\nExisting Python files:")
    for file in sorted(SRC_DIR.glob("*.py")):
        print(f"- {file.name}")

Source folder exists: True

Existing Python files:
- __init__.py
- models.py
- preprocess.py
- robustness.py
- train.py


In [11]:
for filename in ["models.py", "train.py", "preprocess.py"]:
    file_path = SRC_DIR / filename

    print(f"\n{'=' * 20} {filename} {'=' * 20}")

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8")[:12000])
    else:
        print("File not found.")


==================== models.py ====================

import torch
from torch import nn
from torchvision import models


class AudioCNN(nn.Module):
    def __init__(self, embedding_dim=128, num_classes=8):
        super().__init__()

        blocks = []
        in_ch = 1

        for out_ch in (32, 64, 128):
            blocks += [
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
            in_ch = out_ch

        self.features = nn.Sequential(
            *blocks,
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

        self.embedding = nn.Sequential(
            nn.Linear(128, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.25),
        )

        self.classifier = nn.Linear(embedding_dim, num_classes)

    def encode(self, x):
        return self.embedding(self.features(x))

    def forward(self, x):
     

In [12]:
import sys
import numpy as np
from pathlib import Path

# Make the existing src folder available to this notebook
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import your existing model classes
from models import AudioCNN, VisualCNN, MultimodalModel

print("Model classes imported successfully!")

# Inspect one audio feature and one video feature
sample = manifest.iloc[0]

audio_path = Path(sample["audio_feature"])
video_path = Path(sample["video_feature"])

if not audio_path.is_absolute():
    audio_path = PROJECT_DIR / audio_path

if not video_path.is_absolute():
    video_path = PROJECT_DIR / video_path

audio_feature = np.load(audio_path)
video_feature = np.load(video_path)

print("\nSample emotion:", sample["emotion"])
print("Sample actor:", sample["actor_id"])
print("Audio feature shape:", audio_feature.shape)
print("Video feature shape:", video_feature.shape)
print("Audio data type:", audio_feature.dtype)
print("Video data type:", video_feature.dtype)

Model classes imported successfully!

Sample emotion: neutral
Sample actor: 1
Audio feature shape: (96, 251)
Video feature shape: (8, 224, 224, 3)
Audio data type: float32
Video data type: uint8
